# 06. 검색한 근거를 답변으로 구성하고 검토 후 저장하기

**대상:** Python·LangGraph 기초를 아는 개발자. **목표:** 검색·초안 작성·승인·파일 저장을 구분합니다.
05를 실행하지 않아도 됩니다. 필요한 코드와 가상 데이터가 모두 들어 있습니다.
API 키·LLM·임베딩 모델 없이 기본 실습이 실행됩니다. 출력은 LLM 답변이 아닌 결정적 템플릿입니다.

## 준비

```bash
uv venv .venv
uv pip install --python .venv/bin/python "memtomem[langgraph]==0.6.1" jupyterlab ipykernel
uv run --python .venv/bin/python --no-project jupyter lab
```

Windows에서는 `.venv/Scripts/python.exe`를 사용하세요. 최초 패키지 설치에는 인터넷이 필요합니다.
**Run All** 후 맨 아래 LLM 확장은 `SKIP`되는 것이 기본 정상 동작입니다.

In [ ]:
try:
    import langgraph
    import memtomem
except (ImportError, ModuleNotFoundError) as exc:
    raise RuntimeError(
        '먼저 uv pip install "memtomem[langgraph]==0.6.1" jupyterlab ipykernel 실행 후 '
        '해당 환경의 Python 커널을 선택하세요.'
    ) from exc


## 1. 안전한 실습 경계

실제 홈·프로젝트·API 키 대신 임시 상태를 사용합니다. 노트북이 만드는 파일은 실습 종료 시 삭제합니다.
이 실습의 namespace 필터는 기억 선택 기능이며 인증·인가 시스템은 아닙니다.

In [ ]:
import os
import tempfile
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def isolated_lab():
    """실제 홈/클라이언트 설정 대신 이 실습만의 임시 상태를 사용합니다."""
    original_env = dict(os.environ)
    original_cwd = Path.cwd()
    with tempfile.TemporaryDirectory(prefix="memtomem-beginner-") as directory:
        root = Path(directory).resolve()
        # 프록시·사내 CA 환경에서도 선택형 LLM 셀이 동작하도록 네트워크 설정만 남깁니다.
        allowed = {
            "PATH", "LANG", "LC_ALL", "SYSTEMROOT", "WINDIR", "TMPDIR",
            "HTTP_PROXY", "HTTPS_PROXY", "NO_PROXY",
            "http_proxy", "https_proxy", "no_proxy",
            "SSL_CERT_FILE", "SSL_CERT_DIR", "REQUESTS_CA_BUNDLE",
        }
        clean = {k: v for k, v in original_env.items() if k in allowed}
        clean.update(
            HOME=str(root), USERPROFILE=str(root),
            XDG_CONFIG_HOME=str(root / "config"),
            XDG_DATA_HOME=str(root / "data"),
            XDG_STATE_HOME=str(root / "state"),
            XDG_CACHE_HOME=str(root / "cache"),
            MEMTOMEM_FASTEMBED_CACHE=str(root / "cache" / "models"),
            LANGSMITH_TRACING="false", LANGCHAIN_TRACING_V2="false",
        )
        try:
            os.environ.clear()
            os.environ.update(clean)
            os.chdir(root)
            yield root
        finally:
            os.chdir(original_cwd)
            os.environ.clear()
            os.environ.update(original_env)


## 2. 검색 → 초안 → 승인 분기

`MemtomemStore`는 Core의 Markdown 작성·SQLite 색인·검색을 쓰는 고수준 어댑터입니다.
05의 JSON 파일 기반 `MemtomemBaseStore`와 구별하세요. 이 어댑터는 노드가 명시적으로 사용하며 `compile(store=...)` 인자가 아닙니다.

기본 임베딩 공급자를 `none`으로 지정해 BM25 키워드 검색만 수행합니다. 그래서 seed와 같은 어휘인 `Retry policy`로 검색합니다.
빈 검색 결과는 오류입니다. 근거를 찾지 못한 경우 LLM으로 그럴듯한 답을 만드는 흐름이 아닙니다.
`approved=False`면 초안까지만 만들고 종료합니다. 테스트의 `True`는 사용자 승인을 흉내 내는 입력이며, LLM의 자동 승인이 아닙니다.

In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph
from memtomem.integrations.langgraph import MemtomemStore

SEED = "Retry policy: retry at most 5 times with 250 ms backoff and jitter."
QUERY = "Retry policy"
NAMESPACE = "onboarding"

def open_store(root):
    return MemtomemStore(config_overrides={
        "storage": {"sqlite_path": str(root / "index.db")},
        "indexing": {
            "memory_dirs": [str(root / "notes")],
            "project_memory_dirs": [], "auto_discover": False,
            "startup_backfill": False, "extract_entities": False,
        },
        "embedding": {"provider": "none", "dimension": 0},
        "rerank": {"enabled": False},
        "llm": {"enabled": False},
        "search": {"tokenizer": "unicode61"},
    })

class ResearchState(TypedDict, total=False):
    query: str
    context: list[dict]
    draft: str
    approved: bool
    saved_to: str

def build_graph(store, composer=None):
    async def retrieve(state):
        hits = await store.search(
            state["query"], namespace=NAMESPACE, tag_filter="decision",
            bm25_weight=1.0, dense_weight=0.0,
        )
        if not hits:
            raise ValueError("검색 결과 없음: seed와 검색어의 실제 단어를 확인하세요.")
        return {"context": hits}

    async def compose(state):
        if composer is None:
            draft = "releasehandoff: " + state["context"][0]["content"]
        else:
            draft = await composer(state["context"])
        if not draft.strip():
            raise ValueError("빈 답변은 저장하지 않습니다.")
        return {"draft": draft}

    async def save(state):
        # 조건부 edge 외에도 저장 노드 자체에서 승인을 확인합니다.
        if not state.get("approved"):
            raise ValueError("명시적 승인 없이는 저장하지 않습니다.")
        result = await store.add(
            state["draft"], tags=["reviewed"], namespace=NAMESPACE,
            file=str(Path("notes") / "handoff.md"),
        )
        if result.get("error") or result.get("indexed_chunks", 0) <= 0:
            raise RuntimeError(f"저장/색인 실패: {result}")
        return {"saved_to": result["file"]}

    builder = StateGraph(ResearchState)
    builder.add_node("retrieve", retrieve)
    builder.add_node("compose", compose)
    builder.add_node("save", save)
    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "compose")
    builder.add_conditional_edges(
        "compose", lambda state: "save" if state.get("approved") else END
    )
    builder.add_edge("save", END)
    # MemtomemStore는 BaseStore가 아니므로 compile(store=store)에 넣지 않습니다.
    return builder.compile()


## 3. 검증 실행

가상의 재시도 정책 한 문장을 저장하고 찾아옵니다. `indexed_chunks > 0`와 실제 Markdown 파일을 함께 확인합니다.
검색 score·UUID·전체 임시 경로는 실행마다 달라질 수 있어 정확한 문자열로 비교하지 않습니다.

In [ ]:
async def seed_store(store):
    result = await store.add(SEED, tags=["decision"], namespace=NAMESPACE, file=str(Path("notes") / "decision.md"))
    assert "error" not in result and result["indexed_chunks"] > 0

async def demonstrate(root):
    store = open_store(root)
    try:
        assert await store.search(QUERY, namespace=NAMESPACE) == []
        await seed_store(store)
        graph = build_graph(store)

        preview = await graph.ainvoke({"query": QUERY, "approved": False})
        assert preview["context"] and "saved_to" not in preview
        assert not (root / "notes" / "handoff.md").exists()
        print("PASS 미승인: 근거 조회·답변 구성만 실행")

        # 화면의 preview["draft"]를 확인한 사용자가 승인했다는 결정적 테스트입니다.
        print("검토할 초안:", preview["draft"])
        saved = await graph.ainvoke({"query": QUERY, "approved": True})
        source = Path(saved["saved_to"]).resolve()
        assert source.is_relative_to((root / "notes").resolve())
        assert source.is_file() and "releasehandoff" in source.read_text()
        print("PASS 승인: Markdown 원본과 색인 확인")

        assert await store.search(QUERY, namespace="other-project") == []
        assert await store.search(QUERY, namespace=NAMESPACE, tag_filter="absent") == []
        print("PASS namespace·태그 불일치: 검색 결과 없음")

        try:
            await graph.ainvoke({"query": "unmatchedneedlexyz", "approved": True})
        except ValueError as exc:
            assert "검색 결과 없음" in str(exc)
        else:
            raise AssertionError("근거 없는 결과를 성공으로 처리했습니다.")
        print("PASS 검색 실패: 저장하지 않고 오류")
    finally:
        await store.close()

    reopened = open_store(root)
    try:
        hits = await reopened.search(
            "releasehandoff", namespace=NAMESPACE, tag_filter="reviewed"
        )
        assert hits and any("releasehandoff" in hit["content"] for hit in hits)
        assert any(Path(hit["source"]).resolve() == source for hit in hits)
        print("PASS 새 store: 저장한 결과와 원본 경로 재조회")
    finally:
        await reopened.close()

with isolated_lab() as lab_root:
    await demonstrate(lab_root)
assert not lab_root.exists()
print("PASS 임시 상태 정리")


## 4. 해석·실패 복구·연습

`PASS` 6줄과 검토용 초안이 나오면 성공입니다. 처음에는 기록이 없고, 저장 후 근거를 찾으며,
미승인 상태에서는 `handoff.md`가 생기지 않습니다. store를 닫았다 다시 열어도 같은 파일과 결과를 찾습니다.

- 검색 실패: 임베딩 없는 실습이므로 seed와 겹치는 단어를 사용하세요.
- 저장 실패: `error`, `indexed_chunks`, 원본 경로를 확인하세요. force 옵션으로 보호를 우회하지 마세요.
- 커널 중단: 커널을 다시 시작하고 Run All을 실행하세요. 기존 실습 상태를 실제 홈으로 복사하지 마세요.

**변형:** `SEED`의 횟수를 3회로 바꾸고 초안에도 3회가 반영되는지 확인하세요.
**연습:** 검색 결과의 `source`를 인계 초안 끝에 덧붙이세요.
**힌트:** `compose` 노드의 `state["context"][0]["source"]`를 사용합니다.
실제 애플리케이션에서는 사용자가 본 정확한 초안에 승인을 묶고 중복 제출·변경된 근거를 처리해야 합니다.

## 5. 선택: 실제 LLM으로 초안 구성 노드만 교체

기본 실습 성공 후에만 실행하세요. 추가 설치는 `uv pip install openai`입니다.
환경 변수 `OPENAI_API_KEY`, `OPENAI_MODEL`과 명시적 `RUN_LLM=True`가 모두 필요합니다.
모델은 계정에서 사용 가능한 Responses API 모델 ID를 직접 지정하며 임의의 최신 모델로 대체하지 않습니다.

실행하면 가상 seed 콘텐츠가 OpenAI로 전송되고 사용 요금이 발생할 수 있습니다.
API 키·개인 자료는 코드 셀이나 공유 출력에 넣지 마세요. `store=False`는 Responses 객체 저장 설정이며
모든 데이터 보존을 해제한다는 보장이 아닙니다.
출력은 **검토용이며 자동 저장하지 않습니다**. 인증·모델·요청 실패는 오류로 표시하며 통과로 세지 않습니다.

[공식 API 안내](https://developers.openai.com/api/docs/quickstart)
· [LangGraph 메모리 설명](https://docs.langchain.com/oss/python/langgraph/add-memory)

In [ ]:
# 실행하려면 사용자가 직접 True로 바꾸고 환경 변수 두 개를 설정하세요.
RUN_LLM = False
# OPENAI_API_KEY: 노트북 코드/출력에 넣지 않습니다.
# OPENAI_MODEL: 본인 계정에서 사용 가능한 Responses API 모델 ID를 명시합니다.

async def llm_preview(api_key, model):
    from openai import AsyncOpenAI

    with isolated_lab() as root:
        store = open_store(root)
        try:
            await seed_store(store)
            async with AsyncOpenAI(
                api_key=api_key, base_url="https://api.openai.com/v1",
                timeout=60.0, max_retries=0,
            ) as client:
                async def compose_with_llm(hits):
                    response = await client.responses.create(
                        model=model,
                        instructions="주어진 가상 프로젝트 근거만 사용해 한국어 인계 요약을 작성하세요.",
                        input="\n".join(hit["content"] for hit in hits),
                        max_output_tokens=500,
                        store=False,
                    )
                    if response.status != "completed" or not response.output_text.strip():
                        raise RuntimeError("LLM 응답이 완료되지 않았거나 비어 있습니다.")
                    return response.output_text

                result = await build_graph(store, compose_with_llm).ainvoke(
                    {"query": QUERY, "approved": False}
                )
                assert "saved_to" not in result
                print("LLM 검토용 초안 (저장 안 함):", result["draft"])
        finally:
            await store.close()

if not RUN_LLM:
    print("SKIP LLM: RUN_LLM=False (기본 실습에는 필요 없음)")
elif not os.environ.get("OPENAI_API_KEY") or not os.environ.get("OPENAI_MODEL"):
    print("SKIP LLM: OPENAI_API_KEY와 OPENAI_MODEL을 먼저 설정하세요.")
else:
    # 요청/인증 실패는 숨기거나 offline 성공으로 바꾸지 않습니다.
    await llm_preview(os.environ["OPENAI_API_KEY"], os.environ["OPENAI_MODEL"])
